In [3]:
import sys
from pathlib import Path
import torch

ROOT = Path.cwd().parent  # notebook/ 的 parent
sys.path.insert(0, str(ROOT))

import numpy as np

from dlphys.analysis.dmft.solver import DMFTConfig, DMFTSolver
from dlphys.analysis.dmft.softmax_moments import estimate_M_from_S
from dlphys.analysis.dmft.observables import residual_memory, corr_length_exp, power_law_alpha
from dlphys.analysis.dmft.stability import finite_diff_jacobian, spectral_radius

# -------------------------
# Benchmark A/B: M(C) sanity
# -------------------------
L = 8
cfg = DMFTConfig(L=L, gamma=0.5, n_mc=50000, mc_seed=0, damping=0.3)
solver = DMFTSolver(cfg)

# pick a simple C: exponential-ish
C = np.exp(-np.arange(L+1)/3.0)
S = solver.compute_S(C)

M = estimate_M_from_S(S, n_mc=80000, seed=0)
print("M symmetry error:", np.max(np.abs(M - M.T)))
print("sum M:", M.sum())

# scaling-degenerate: shrink logits variance -> uniform softmax
S_small = 1e-6 * S
M_small = estimate_M_from_S(S_small, n_mc=120000, seed=0)
M_target = np.ones((L+1, L+1)) / (L+1)**2
print("uniform M error:", np.max(np.abs(M_small - M_target)))
print("sum M_small:", M_small.sum())

# -------------------------
# Benchmark C: fixed point iteration
# -------------------------
def run_once(C0):
    out = solver.iterate(C0, verbose=False)
    Cstar = out["C_star"]
    return out["converged"], Cstar, out["err_hist"]

C0a = 0.2 * np.ones(L+1)
C0b = np.exp(-np.arange(L+1)/2.0)

conv_a, Cstar_a, err_a = run_once(C0a)
conv_b, Cstar_b, err_b = run_once(C0b)

print("converged A/B:", conv_a, conv_b)
print("Cstar distance:", np.max(np.abs(Cstar_a - Cstar_b)))

print("q:", residual_memory(Cstar_a))
print("xi(exp):", corr_length_exp(Cstar_a))
print("alpha(power):", power_law_alpha(Cstar_a))

# Optional: stability (expensive because F uses MC)
# Make MC smaller just for Jacobian test
cfgJ = DMFTConfig(L=L, gamma=0.5, n_mc=15000, mc_seed=1, damping=0.3, tol=1e-6, max_iter=200)
solverJ = DMFTSolver(cfgJ)
outJ = solverJ.iterate(C0a)
CstarJ = outJ["C_star"]

def F_only(C_in):
    Cnew, _ = solverJ.F(C_in)
    a = cfgJ.damping
    return (1-a)*C_in + a*Cnew

J = finite_diff_jacobian(F_only, CstarJ, eps=5e-5)
rho = spectral_radius(J)
print("rho(J):", rho)

M symmetry error: 0.0
sum M: 0.9999999999999999
uniform M error: 5.4467231653740344e-08
sum M_small: 1.0
converged A/B: True True
Cstar distance: 8.966288550024168e-09
q: 9.804727415030705e-07
xi(exp): 8.795853181613298
alpha(power): 0.3541934084913722


LinAlgError: Matrix is not positive definite